In [ ]:
"""
Project 04: Hospital Readmission
Step 01: EDA and ICD-9 Clinical Mapping
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import time
from tqdm import tqdm

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 01: EDA and ICD-9 Clinical Mapping")
print("=" * 60)
start_time = time.time()

In [ ]:
# ─── 1. Load Dataset ───
print("\n[1/4] Loading dataset...")
df = pd.read_csv(os.path.join(RAW, "diabetic_data.csv"))
print(f"  Initial Dataset Shape: {df.shape}")

In [ ]:
# ─── 2. Basic Cleaning ───
print("[2/4] Cleaning data and handling duplicates...")
df.replace("?", np.nan, inplace=True)
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"  Columns with missing values: {len(missing)}")

In [ ]:
cols_to_drop = ["weight", "payer_code", "medical_specialty"]
df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
print(f"  Dropped high-missing columns: {cols_to_drop}")

In [ ]:
df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
print(f"  Shape after removing duplicate patients: {df.shape}")

In [ ]:
# ─── 3. ICD-9 Clinical Categorization ───
print("[3/4] Mapping ICD-9 codes to clinical categories...")

In [ ]:
def map_icd9_to_group(code):
    if pd.isnull(code):
        return "Other"
    code_str = str(code)
    if "V" in code_str or "E" in code_str:
        return "Other"
    try:
        val = float(code)
        if 390 <= val <= 459 or val == 785:
            return "Circulatory"
        elif 460 <= val <= 519 or val == 786:
            return "Respiratory"
        elif 520 <= val <= 579 or val == 787:
            return "Digestive"
        elif val == 250:
            return "Diabetes"
        elif 800 <= val <= 999:
            return "Injury"
        elif 710 <= val <= 739:
            return "Musculoskeletal"
        elif 580 <= val <= 629 or val == 788:
            return "Genitourinary"
        elif 140 <= val <= 239:
            return "Neoplasms"
        else:
            return "Other"
    except:
        return "Other"

In [ ]:
diag_cols = ["diag_1", "diag_2", "diag_3"]
for col in tqdm(diag_cols, desc="  ICD-9 Mapping"):
    df[f"{col}_group"] = df[col].apply(map_icd9_to_group)

In [ ]:
# Plot primary diagnosis distribution
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(
    y="diag_1_group", data=df, order=df["diag_1_group"].value_counts().index, ax=ax
)
ax.set_title("Distribution of Primary Diagnosis Clinical Categories", fontsize=14)
ax.set_xlabel("Count")
ax.set_ylabel("Diagnosis Group")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "diagnosis_distribution.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"  Saved: charts/diagnosis_distribution.png")

In [ ]:
# ─── 4. Binary Target and Final Save ───
print("[4/4] Creating binary target and saving...")
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)
class_balance = df["readmitted_30d"].value_counts(normalize=True)
print(
    f"  Class Balance: Not Readmitted={class_balance[0]:.1%}, Readmitted={class_balance[1]:.1%}"
)

In [ ]:
output_path = os.path.join(PROCESSED, "readmission_cleaned.csv")
df.to_csv(output_path, index=False)
print(f"  Saved: {output_path}")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(
    f"STEP 01 COMPLETE | Time: {elapsed:.1f}s | Records: {len(df)} | Features: {len(df.columns)}"
)
print(f"{'=' * 60}")